YOLOv8n metrics:

In [ ]:
from ultralytics import YOLO

# путь к весам YOLOv8n
weights = "/Users/zahar/PycharmProjects/RDD_CV/YOLOv8n/project/best.pt"

# путь к датасету
data_yaml = "/Users/zahar/PycharmProjects/RDD_CV/road_damage_dataset_3000/data.yaml"

# загрузка модели
model = YOLO(weights)

# валидация
metrics = model.val(data=data_yaml, split="val", verbose=False)

# --- основные метрики ---
precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map5095 = metrics.box.map

# F1-score (средний)
f1 = 2 * precision * recall / (precision + recall + 1e-16)

# вывод
print("===== YOLOv8n =====")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"F1-score      : {f1:.4f}")
print(f"mAP@0.5       : {map50:.4f}")
print(f"mAP@0.5:0.95  : {map5095:.4f}")

YOLO11n metrics:

In [ ]:
from ultralytics import YOLO

# путь к весам YOLOv8n
weights = "/Users/zahar/PycharmProjects/RDD_CV/YOLO11n/project/best.pt"

# путь к датасету
data_yaml = "/Users/zahar/PycharmProjects/RDD_CV/road_damage_dataset_3000/data.yaml"

# загрузка модели
model = YOLO(weights)

# валидация
metrics = model.val(data=data_yaml, split="val", verbose=False)

# --- основные метрики ---
precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map5095 = metrics.box.map

# F1-score (средний)
f1 = 2 * precision * recall / (precision + recall + 1e-16)

# вывод
print("===== YOLO11n =====")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"F1-score      : {f1:.4f}")
print(f"mAP@0.5       : {map50:.4f}")
print(f"mAP@0.5:0.95  : {map5095:.4f}")

YOLO26n metrics:

In [ ]:
from ultralytics import YOLO

# путь к весам YOLOv8n
weights = "/Users/zahar/PycharmProjects/RDD_CV/YOLO26n/project/best.pt"

# путь к датасету
data_yaml = "/Users/zahar/PycharmProjects/RDD_CV/road_damage_dataset_3000/data.yaml"

# загрузка модели
model = YOLO(weights)

# валидация
metrics = model.val(data=data_yaml, split="val", verbose=False)

# --- основные метрики ---
precision = metrics.box.mp
recall = metrics.box.mr
map50 = metrics.box.map50
map5095 = metrics.box.map

# F1-score (средний)
f1 = 2 * precision * recall / (precision + recall + 1e-16)

# вывод
print("===== YOLO26n =====")
print(f"Precision     : {precision:.4f}")
print(f"Recall        : {recall:.4f}")
print(f"F1-score      : {f1:.4f}")
print(f"mAP@0.5       : {map50:.4f}")
print(f"mAP@0.5:0.95  : {map5095:.4f}")

SSD metrics:

In [ ]:
# ======================================
# 0. IMPORTS
# ======================================
import torch
import torchvision
import os
import cv2
from torchvision.models.detection.ssd import SSDClassificationHead
from torch.utils.data import Dataset, DataLoader
import yaml
import numpy as np
from tqdm import tqdm

# ======================================
# 1. DEVICE
# ======================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ======================================
# 2. PATHS
# ======================================
DATASET_PATH = "/Users/zahar/PycharmProjects/RDD_CV/road_damage_dataset_3000"
MODEL_PATH = "/Users/zahar/PycharmProjects/RDD_CV/SSD/project/best_model.pth"

# ======================================
# 3. LOAD YAML
# ======================================
with open(f"{DATASET_PATH}/data.yaml", "r") as f:
    data = yaml.safe_load(f)

NUM_CLASSES = len(data["names"]) + 1
print("Classes:", NUM_CLASSES)

# ======================================
# 4. DATASET
# ======================================
class YOLODataset(Dataset):
    def __init__(self, img_dir, label_dir):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.images = [f for f in os.listdir(img_dir) if f.endswith(".jpg") or f.endswith(".png")]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (300, 300))
        h, w, _ = img.shape

        label_path = os.path.join(self.label_dir, img_name.replace(".jpg", ".txt").replace(".png", ".txt"))

        boxes = []
        labels = []

        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls, x, y, bw, bh = map(float, parts)
                    x1 = (x - bw/2) * w
                    y1 = (y - bh/2) * h
                    x2 = (x + bw/2) * w
                    y2 = (y + bh/2) * h
                    boxes.append([x1, y1, x2, y2])
                    labels.append(int(cls) + 1)

        img = torch.from_numpy(img).float().permute(2,0,1) / 255.0
        boxes = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0,4), dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor([idx])}
        return img, target

# ======================================
# 5. DATALOADER
# ======================================
val_dataset = YOLODataset(f"{DATASET_PATH}/valid/images", f"{DATASET_PATH}/valid/labels")

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=lambda x: tuple(zip(*x))
)

print("Validation images:", len(val_dataset))

# ======================================
# 6. LOAD MODEL
# ======================================
model = torchvision.models.detection.ssd300_vgg16(weights=None, weights_backbone=None)

from torchvision.models.detection.ssd import SSDClassificationHead
in_channels = [512, 1024, 512, 256, 256, 256]
num_anchors = model.anchor_generator.num_anchors_per_location()

model.head.classification_head = SSDClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=NUM_CLASSES
)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()
print("SSD loaded successfully")

# ======================================
# 7. IOU FUNCTION
# ======================================
def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0])*(box1[3]-box1[1])
    area2 = (box2[2]-box2[0])*(box2[3]-box2[1])
    union = area1 + area2 - inter + 1e-6
    return inter / union

# ======================================
# 8. EVALUATION
# ======================================
TP = 0
FP = 0
FN = 0
all_ious = []

with torch.no_grad():
    for images, targets in tqdm(val_loader):
        images = [img.to(device) for img in images]
        outputs = model(images)

        for i in range(len(outputs)):
            pred_boxes = outputs[i]["boxes"].cpu()
            pred_scores = outputs[i]["scores"].cpu()
            gt_boxes = targets[i]["boxes"].cpu()

            keep = pred_scores > 0.5
            pred_boxes = pred_boxes[keep]

            matched_gt = set()
            for pred_box in pred_boxes:
                best_iou = 0
                best_gt = -1
                for j, gt_box in enumerate(gt_boxes):
                    if j in matched_gt:
                        continue
                    current_iou = iou(pred_box.numpy(), gt_box.numpy())
                    if current_iou > best_iou:
                        best_iou = current_iou
                        best_gt = j
                if best_iou >= 0.5:
                    TP += 1
                    matched_gt.add(best_gt)
                    all_ious.append(best_iou)
                else:
                    FP += 1
            FN += len(gt_boxes) - len(matched_gt)

precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)
f1 = 2*precision*recall/(precision+recall+1e-6)
mAP50 = np.mean(all_ious) if len(all_ious)>0 else 0

print("\n================ SSD METRICS ================")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"mAP@0.5   : {mAP50:.4f}")
print("=============================================")

Faster R-CNN metrics:

In [ ]:
import os
import torch
import torchvision
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F

# =========================
# DEVICE
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# PATHS
# =========================
DATASET_PATH = "/Users/zahar/PycharmProjects/RDD_CV/road_damage_dataset_3000"
MODEL_PATH = "/Users/zahar/PycharmProjects/RDD_CV/Faster R-CNN/project/best_model.pth"

# =========================
# DATASET
# =========================
class YOLODataset(Dataset):
    def __init__(self, img_dir, label_dir):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.images = [f for f in os.listdir(img_dir) if f.endswith(".jpg") or f.endswith(".png")]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)

        img = Image.open(img_path).convert("RGB")
        w, h = img.size

        label_path = os.path.join(
            self.label_dir,
            img_name.replace(".jpg", ".txt").replace(".png", ".txt")
        )

        boxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f:
                    cls, x, y, bw, bh = map(float, line.split())

                    x1 = (x - bw / 2) * w
                    y1 = (y - bh / 2) * h
                    x2 = (x + bw / 2) * w
                    y2 = (y + bh / 2) * h

                    boxes.append([x1, y1, x2, y2])
                    labels.append(int(cls) + 1)

        img = F.to_tensor(img)

        return img, {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64)
        }

def collate_fn(batch):
    return tuple(zip(*batch))

val_dataset = YOLODataset(
    f"{DATASET_PATH}/valid/images",
    f"{DATASET_PATH}/valid/labels"
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)

print("Val images:", len(val_dataset))

# =========================
# LOAD CHECKPOINT SAFELY
# =========================
checkpoint = torch.load(MODEL_PATH, map_location=device)

# иногда сохраняют dict внутри dict
if "model" in checkpoint:
    checkpoint = checkpoint["model"]

# =========================
# BUILD MODEL AUTOMATICALLY
# =========================
# УМНО: берём число классов из веса
num_classes = None

for k, v in checkpoint.items():
    if "box_predictor.cls_score.bias" in k:
        num_classes = v.shape[0]
        break

# fallback
if num_classes is None:
    num_classes = 91  # COCO fallback

print("Detected classes in checkpoint:", num_classes)

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=None)

# FIX HEAD (must match checkpoint)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
    in_features,
    num_classes
)

# =========================
# LOAD WEIGHTS SAFELY (NO CRASH)
# =========================
missing, unexpected = model.load_state_dict(checkpoint, strict=False)

print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

model.to(device)
model.eval()

print("Model loaded safely")

# =========================
# IoU
# =========================
def iou(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = (a[2]-a[0]) * (a[3]-a[1])
    area2 = (b[2]-b[0]) * (b[3]-b[1])

    return inter / (area1 + area2 - inter + 1e-6)

# =========================
# EVAL (SAFE)
# =========================
TP, FP, FN = 0, 0, 0
ious = []

with torch.no_grad():

    for i, (images, targets) in enumerate(val_loader):

        if i % 20 == 0:
            print(f"Processing batch {i}/{len(val_loader)}")

        images = [img.to(device) for img in images]
        outputs = model(images)

        for pred, gt in zip(outputs, targets):

            pred_boxes = pred["boxes"].cpu()
            scores = pred["scores"].cpu()
            gt_boxes = gt["boxes"]

            keep = scores > 0.5
            pred_boxes = pred_boxes[keep]

            matched = set()

            for pb in pred_boxes:

                best_iou = 0
                best_j = -1

                for j, gb in enumerate(gt_boxes):

                    if j in matched:
                        continue

                    score = iou(pb, gb)

                    if score > best_iou:
                        best_iou = score
                        best_j = j

                if best_iou >= 0.5:
                    TP += 1
                    matched.add(best_j)
                else:
                    FP += 1

            FN += len(gt_boxes) - len(matched)

# =========================
# METRICS
# =========================
precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)
f1 = 2 * precision * recall / (precision + recall + 1e-6)
mAP50 = np.mean(ious) if len(ious) > 0 else 0

print("\n================ FINAL RESULTS ================")
print(f"TP: {TP} | FP: {FP} | FN: {FN}")
print("----------------------------------------------")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"mAP@0.5   : {mAP50:.4f}")
print("==============================================")

DETR metrics:

In [ ]:
# ======================================
# DETR EVALUATION (FIXED VERSION)
# ======================================

import torch
import os
import cv2
import yaml
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import DetrImageProcessor, DetrForObjectDetection
from tqdm import tqdm

# ======================================
# DEVICE
# ======================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ======================================
# PATHS
# ======================================
DATASET_PATH = "/Users/zahar/PycharmProjects/RDD_CV/road_damage_dataset_3000"
MODEL_PATH = "/Users/zahar/PycharmProjects/RDD_CV/DETR/project/best_model.pth"

# ======================================
# LOAD DATA INFO
# ======================================
with open(f"{DATASET_PATH}/data.yaml", "r") as f:
    data = yaml.safe_load(f)

CLASS_NAMES = data["names"]
NUM_CLASSES = len(CLASS_NAMES)

print("Classes:", CLASS_NAMES)

# ======================================
# MODEL + PROCESSOR
# ======================================
processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")

model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

print("DETR loaded successfully!")

# ======================================
# DATASET
# ======================================
class YOLODataset(Dataset):
    def __init__(self, img_dir, label_dir):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.images = [
            f for f in os.listdir(img_dir)
            if f.endswith(".jpg") or f.endswith(".png")
        ]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, (800, 800))

        h, w = 800, 800

        label_path = os.path.join(
            self.label_dir,
            img_name.replace(".jpg", ".txt").replace(".png", ".txt")
        )

        boxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f:
                    cls, x, y, bw, bh = map(float, line.split())

                    x1 = (x - bw/2) * w
                    y1 = (y - bh/2) * h
                    x2 = (x + bw/2) * w
                    y2 = (y + bh/2) * h

                    boxes.append([x1, y1, x2, y2])
                    labels.append(int(cls))

        return {
            "image": image,
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "img_name": img_name
        }

# ======================================
# COLLATE FN (🔥 FIX HERE)
# ======================================
def collate_fn(batch):
    return {
        "images": [b["image"] for b in batch],
        "boxes": [b["boxes"] for b in batch],
        "labels": [b["labels"] for b in batch],
        "img_name": [b["img_name"] for b in batch],
    }

# ======================================
# VAL LOADER
# ======================================
val_dataset = YOLODataset(
    f"{DATASET_PATH}/valid/images",
    f"{DATASET_PATH}/valid/labels"
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)

print("Validation images:", len(val_dataset))

# ======================================
# IoU FUNCTION
# ======================================
def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    union = area1 + area2 - inter + 1e-6
    return inter / union

# ======================================
# METRICS
# ======================================
TP, FP, FN = 0, 0, 0
all_ious = []

with torch.no_grad():

    for batch in tqdm(val_loader):

        images = batch["images"]

        pixel_values = processor(images=images, return_tensors="pt").pixel_values.to(device)

        outputs = model(pixel_values=pixel_values)

        target_sizes = torch.tensor([[800, 800]] * len(images)).to(device)

        results = processor.post_process_object_detection(
            outputs,
            threshold=0.5,
            target_sizes=target_sizes
        )

        for i in range(len(results)):

            pred_boxes = results[i]["boxes"].cpu().numpy()
            pred_scores = results[i]["scores"].cpu().numpy()
            pred_labels = results[i]["labels"].cpu().numpy()

            gt_boxes = batch["boxes"][i].numpy()
            gt_labels = batch["labels"][i].numpy()

            matched = set()

            # -------------------------
            # MATCHING
            # -------------------------
            for pb, pl in zip(pred_boxes, pred_labels):

                best_iou = 0
                best_j = -1

                for j, gb in enumerate(gt_boxes):

                    if j in matched:
                        continue

                    if pl != gt_labels[j]:
                        continue

                    current_iou = iou(pb, gb)

                    if current_iou > best_iou:
                        best_iou = current_iou
                        best_j = j

                if best_iou >= 0.5:
                    TP += 1
                    matched.add(best_j)
                    all_ious.append(best_iou)
                else:
                    FP += 1

            FN += len(gt_boxes) - len(matched)

# ======================================
# FINAL METRICS
# ======================================
precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)
f1 = 2 * precision * recall / (precision + recall + 1e-6)

mAP50 = np.mean(all_ious) if len(all_ious) > 0 else 0

print("\n================ DETR METRICS ================")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"mAP@0.5   : {mAP50:.4f}")
print("===============================================")